# PHAN TICH DU LIEU TUYEN DUNG IT TAI VIET NAM

Notebook nay trinh bay quy trinh thu thap du lieu, thong ke mo ta, cac buoc tien xu ly du lieu va khung truc quan hoa cho bai toan du bao muc luong `salary_avg`.

## 1. Data and Setup

Du lieu duoc crawl tu hai nguon la `ITViec` va `TopCV`. Sau khi thu thap, du lieu raw se duoc merge, lam sach, chuan hoa salary, suy ra mot so bien category va trich xuat skill phuc vu EDA va modeling.

In [ ]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import Markdown, display

from src.processing.clean_jobs import (
    clean_jobs,
    compute_salary_fields,
    normalize_company_type,
    normalize_level,
    normalize_location,
    normalize_remote_option,
    normalize_text,
    parse_experience_years,
)
from src.processing.extract_skills import enrich_with_skill_features, extract_skills_from_row

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
CLEAN_DIR = BASE_DIR / 'data' / 'clean-data'

raw_itviec_path = RAW_DIR / 'itviec_jobs_20260315_115039.csv'
raw_topcv_path = RAW_DIR / 'topcv_jobs_20260314_231436.csv'
clean_path = CLEAN_DIR / 'jobs_cleaned.csv'
features_path = CLEAN_DIR / 'jobs_features.csv'

df_itviec = pd.read_csv(raw_itviec_path)
df_topcv = pd.read_csv(raw_topcv_path)
df_clean = pd.read_csv(clean_path)
df_features = pd.read_csv(features_path)


## 2. Trinh bay ma nguon crawl du lieu

Phan nay trich dan truc tiep ma nguon crawler de minh hoa cach du lieu duoc thu thap tu tung website.

In [ ]:
def show_code(path_str, max_lines=None):
    path = BASE_DIR / path_str
    lines = path.read_text(encoding='utf-8').splitlines()
    if max_lines is not None:
        lines = lines[:max_lines]
    code_block = '```python\n' + '\n'.join(lines) + '\n```'
    display(Markdown(f'### `{path_str}`\n\n' + code_block))


**Crawler 1: ITViec**

Crawler nay thuc hien parse thong tin tu trang job detail va lay cac truong chinh nhu `job_title`, `tech_stack`, `experience_years`, `location`, `salary_min`, `salary_max`, `currency`.

In [ ]:
show_code('src/data_collection/itviec_crawler_update.py', max_lines=220)

**Crawler 2: TopCV**

Crawler TopCV tap trung thu thap danh sach tin tuyen dung, dong thoi rut trich salary va skill tu phan hien thi tren job card.

In [ ]:
show_code('src/data_collection/topcv_crawler.py', max_lines=200)

## 3. Tong quan du lieu thu thap

Truoc khi cleaning, can kiem tra kich thuoc du lieu, schema va mot vai mau dai dien cua tung nguon.

In [ ]:
summary_df = pd.DataFrame({
    'dataset': ['ITViec raw', 'TopCV raw', 'Merged clean', 'Feature dataset'],
    'rows': [len(df_itviec), len(df_topcv), len(df_clean), len(df_features)],
    'columns': [df_itviec.shape[1], df_topcv.shape[1], df_clean.shape[1], df_features.shape[1]],
})
summary_df

**Schema cua 2 file raw**

In [ ]:
print('ITViec columns:', list(df_itviec.columns))
print('TopCV columns:', list(df_topcv.columns))

**Mau du lieu th?**

In [ ]:
display(df_itviec.head(3))
display(df_topcv.head(3))

## 4. Thong ke mo ta du lieu

Phan nay tap trung vao thong ke co ban cua bo du lieu sau cleaning de danh gia muc do day du va muc do phu hop cho bai toan du bao `salary_avg`.

**Thong tin tong quan cua bo du lieu clean**

In [ ]:
df_clean.info()

**Thong ke gia tri thieu**

In [ ]:
missing_summary = df_clean.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df_clean)).round(4)
missing_summary

**Thong ke mo ta cac bien dinh luong**

In [ ]:
salary_cols = ['salary_min', 'salary_max', 'salary_avg', 'experience_years']
df_clean[salary_cols].describe().T

**So luong tin theo nguon du lieu**

In [ ]:
df_clean['source'].value_counts().to_frame('job_count')

**Top dia diem co nhieu tin tuyen dung nhat**

In [ ]:
df_clean['location'].value_counts().head(10).to_frame('job_count')

**Top cap bac duoc suy ra tu title va experience**

In [ ]:
df_clean['level'].value_counts(dropna=False).to_frame('job_count')

**Ty le record co va khong co salary_avg**

In [ ]:
salary_availability = pd.Series({
    'co_salary_avg': int(df_clean['salary_avg'].notna().sum()),
    'thieu_salary_avg': int(df_clean['salary_avg'].isna().sum()),
}).to_frame('count')
salary_availability['ratio'] = (salary_availability['count'] / len(df_clean)).round(4)
salary_availability

## 5. Nhan xet ban dau tu thong ke mo ta

- Sau khi gop 2 nguon, bo du lieu co the dung cho EDA va bai toan hoi quy du bao `salary_avg`.
- Khong phai moi tin tuyen dung deu cong khai l??ng, vi vay `salary_avg` co ty le thieu nhat dinh.
- Cac cot `location`, `level`, `company_type`, `remote_option`, `tech_stack` va nhom skill la cac dac trung co kha nang tac dong den salary.
- Viec xu ly missing salary can duoc lam than trong: giu lai record cho EDA nhung loai khoi tap train neu khong co target.

## 6. Tien xu ly du lieu

Phan nay trinh bay chi tiet tung buoc tien xu ly du lieu. Moi buoc deu co muc tieu ro rang, code su dung va vi du minh hoa de giai thich tai sao buoc do can thiet.

### 6.1. Muc tieu cua qua trinh cleaning

Muc tieu cua cleaning trong de tai nay khong chi la xoa null hay bo trung lap. Quan trong hon, no can bien du lieu raw thanh mot bang nhat quan de co the:

- tao dung bien muc tieu `salary_avg`
- so sanh salary giua cac nguon tren cung mot don vi
- bien cac truong text thanh cac bien category co the phan tich
- giu lai nhung record van con gia tri cho EDA, ke ca khi thieu salary
- tao mot bo du lieu sach de lam nen cho feature engineering va modeling

### 6.2. Gop du lieu raw

Hai file raw CSV duoc doc va noi thanh mot bang duy nhat. Trong qua trinh nay, cot `source` duoc them vao de biet moi record den tu `itviec` hay `topcv`.

In [ ]:
df_raw_merged = pd.concat([
    df_itviec.assign(source='itviec'),
    df_topcv.assign(source='topcv')
], ignore_index=True)
df_raw_merged.head(3)

### 6.3. Lam sach text co ban

Cac cot text thuong co khoang trang thua, chuoi rong hoac bieu dien khong dong nhat. Ham `normalize_text()` duoc dung de:

- trim khoang trang dau va cuoi chuoi
- quy tat nhieu khoang trang lien tiep ve mot khoang trang
- bien chuoi rong thanh gia tri thieu (`NaN`)

Buoc nay duoc ap dung cho cac cot: `job_title`, `company_name`, `location`, `remote_option`, `tech_stack`.

In [ ]:
print(inspect.getsource(normalize_text))

sample_text = pd.Series(['  Ha Noi  ', 'Remote   only', '', None])
pd.DataFrame({
    'before': sample_text,
    'after': sample_text.apply(normalize_text)
})

### 6.4. Chuan hoa location va remote option

Du lieu raw co nhieu bien the cho cung mot noi dung. Vi du `H? N?i`, `Ha Noi`, `Hanoi` can duoc dua ve mot gia tri chung de tranh tan man nhom khi thong ke.

- `normalize_location()` dung bang anh xa de dua cac bien the ve mot chuan
- `normalize_remote_option()` thong nhat `onsite`, `remote`, `hybrid`
- Neu 1 tin co nhieu dia diem, cac dia diem duoc noi bang dau `|`

In [ ]:
print(inspect.getsource(normalize_location))

location_demo = pd.DataFrame({
    'raw_location': ['H? N?i', 'Ha Noi', 'HCM', 'TP.HCM', 'H? N?i & H? Ch? Minh (m?i)'],
    'normalized_location': [
        normalize_location('H? N?i'),
        normalize_location('Ha Noi'),
        normalize_location('HCM'),
        normalize_location('TP.HCM'),
        normalize_location('H? N?i & H? Ch? Minh (m?i)')
    ]
})
location_demo

In [ ]:
remote_demo = pd.DataFrame({
    'raw_remote_option': ['on-site', 'onsite', 'office', 'WFH', 'work from home', 'hybrid'],
    'normalized_remote_option': [
        normalize_remote_option('on-site'),
        normalize_remote_option('onsite'),
        normalize_remote_option('office'),
        normalize_remote_option('WFH'),
        normalize_remote_option('work from home'),
        normalize_remote_option('hybrid')
    ]
})
remote_demo

### 6.5. Chuan hoa experience

Cot `experience_years` trong raw data co the xuat hien duoi dang text nhu `1 years`, `2`, `3+ years`. Ham `parse_experience_years()` tach gia tri so tu chuoi va quy ve dang so de de phan tich va train model.

In [ ]:
print(inspect.getsource(parse_experience_years))

experience_demo = pd.DataFrame({
    'raw_experience': ['1 years', '2', '3+ years', 'At least 5 years', None],
    'parsed_experience': [
        parse_experience_years('1 years'),
        parse_experience_years('2'),
        parse_experience_years('3+ years'),
        parse_experience_years('At least 5 years'),
        parse_experience_years(None)
    ]
})
experience_demo

### 6.6. Xu ly salary va tao `salary_avg`

Day la buoc quan trong nhat trong cleaning vi `salary_avg` la bien muc tieu cua bai toan. Pipeline xu ly salary theo cac nguyen tac sau:

- chuyen `salary_min`, `salary_max` ve dang so
- xac dinh `salary_currency`; neu thieu thi suy ra theo nguon (`itviec` -> `USD`, `topcv` -> `VND`)
- quy doi ve cung mot don vi `VND/thang`
- neu `salary_min > salary_max` thi dao lai
- neu chi co mot dau mut luong thi lay chinh gia tri do de uoc luong dau con lai va gan co `salary_is_estimated = 1`
- neu thieu ca hai dau mut thi giu `NaN`, khong dien `0`
- tao `salary_avg = mean(salary_min, salary_max)`
- loai bo cac gia tri salary bat hop ly sau quy doi

In [ ]:
print(inspect.getsource(compute_salary_fields))

In [ ]:
salary_demo = df_raw_merged[['source', 'salary_min', 'salary_max', 'currency']].head(8).copy()
salary_demo_before = salary_demo.copy()
salary_demo_after = compute_salary_fields(salary_demo)
pd.concat([
    salary_demo_before.add_prefix('before_'),
    salary_demo_after[['salary_min', 'salary_max', 'salary_avg', 'salary_currency', 'salary_is_estimated']].add_prefix('after_')
], axis=1)

### 6.7. Xu ly record thieu l??ng

Voi cac record khong co du lieu salary, pipeline khong loai bo ngay o buoc cleaning. Thay vao do:

- giu lai record trong `jobs_cleaned.csv` de van dung duoc cho EDA
- dat `salary_min`, `salary_max`, `salary_avg` la `NaN` neu khong co du lieu
- chi loai khoi tap train khi xay dung mo hinh hoi quy, vi khi do moi can target `Y = salary_avg`

Cach xu ly nay giup tranh tao du lieu gia va van bao ton thong tin thi truong tuyen dung.

In [ ]:
df_clean[df_clean['salary_avg'].isna()].head(5)

### 6.8. Tao bien category moi: `level` va `company_type`

Sau khi da co `job_title` va `experience_years` duoc lam sach, pipeline suy ra them mot so bien category co y nghia:

- `level`: Intern, Junior, Middle, Senior, Lead, Manager
- `company_type`: Finance, Education, Healthcare, Technology, Logistics, Real Estate, Other

Muc tieu cua buoc nay la bien doi thong tin text thanh cac nhom de de tong hop, truc quan hoa va dua vao model.

In [ ]:
print(inspect.getsource(normalize_level))

level_demo = pd.DataFrame({
    'job_title': ['Junior Python Developer', 'Senior Backend Engineer', 'QA Manager', 'Data Engineer'],
    'experience_years': [1, 5, 6, 3],
    'level': [
        normalize_level('Junior Python Developer', 1),
        normalize_level('Senior Backend Engineer', 5),
        normalize_level('QA Manager', 6),
        normalize_level('Data Engineer', 3),
    ]
})
level_demo

In [ ]:
company_demo = pd.DataFrame({
    'company_name': [
        'Techcombank',
        'ABC Software Solutions',
        'Giao H?ng Ti?t Ki?m',
        'XYZ Academy',
        'Some Unknown Company'
    ]
})
company_demo['company_type'] = company_demo['company_name'].apply(normalize_company_type)
company_demo

### 6.9. Loai bo duplicate va tao `job_id`

Sau khi da chuan hoa du lieu, pipeline tao `job_id` tu cac truong nhan dien chinh va xoa cac record trung lap. Buoc nay giup tranh dem lap cung mot tin tuyen dung nhieu lan khi thong ke.

In [ ]:
raw_count = len(df_raw_merged)
clean_count = len(df_clean)
pd.DataFrame({
    'metric': ['raw_rows_after_merge', 'rows_after_cleaning', 'removed_rows'],
    'value': [raw_count, clean_count, raw_count - clean_count]
})

### 6.10. Feature engineering tu skill

Sau buoc cleaning, script `extract_skills.py` duoc chay rieng de tao `jobs_features.csv`. Buoc nay khong lam thay doi file clean, ma chi bo sung them cac feature tu `tech_stack` va `job_title`.

Cac viec chinh duoc thuc hien:

- tach danh sach skill tu `tech_stack`
- chuan hoa alias skill nhu `nodejs` -> `node.js`, `javascript` -> `js`
- tao cot `skills_extracted`
- tao `skills_count`
- tao cac cot nhi phan nhu `has_python`, `has_sql`, `has_cloud`, `has_devops`

In [ ]:
print(inspect.getsource(extract_skills_from_row))

In [ ]:
feature_preview = enrich_with_skill_features(df_clean.head(5).copy())
feature_preview[['job_title', 'tech_stack', 'skills_extracted', 'skills_count', 'has_python', 'has_sql', 'has_cloud', 'has_devops']]

### 6.11. So sanh `jobs_cleaned.csv` va `jobs_features.csv`

Hai file dau ra phuc vu hai muc dich khac nhau:

- `jobs_cleaned.csv`: dung cho kiem tra chat luong du lieu va EDA co ban
- `jobs_features.csv`: dung cho cac buoc feature engineering va modeling

In [ ]:
comparison_df = pd.DataFrame({
    'dataset': ['jobs_cleaned.csv', 'jobs_features.csv'],
    'rows': [len(df_clean), len(df_features)],
    'columns': [df_clean.shape[1], df_features.shape[1]],
    'has_skills_extracted': ['skills_extracted' in df_clean.columns, 'skills_extracted' in df_features.columns],
    'has_skill_indicators': ['has_python' in df_clean.columns, 'has_python' in df_features.columns],
})
comparison_df

## 7. Truc quan hoa du lieu

Phan nay giu khung trinh bay cho cac bieu do, chua dien phan tich chi tiet.

### 7.1. Truc quan hoa don bien

- Phan bo `salary_avg` sau khi cleaning
- So luong tin theo `source`
- So luong tin theo `location`
- So luong tin theo `level`

In [ ]:
# TODO: Ve histogram/boxplot cho salary_avg
# TODO: Ve countplot cho source, location, level

### 7.2. Truc quan hoa hai bien va da bien

- `salary_avg` theo `level`
- `salary_avg` theo `location`
- `salary_avg` theo nhom skill
- So sanh salary giua `remote`, `hybrid`, `onsite`

In [ ]:
# TODO: Ve boxplot/barplot de so sanh salary theo level, location, remote_option
# TODO: Kiem tra moi quan he giua salary_avg va skills_count

### 7.3. Truc quan hoa nhieu chieu

- Heatmap tuong quan giua cac bien so
- Embedding hoac giam chieu cho nhom feature skill neu can
- Tong hop insight tu cac bieu do

In [ ]:
# TODO: Correlation heatmap
# TODO: TSNE/PCA neu can phan tich khong gian feature

## 8. Nhan xet va ket luan tam thoi

- Du lieu da duoc crawl tu 2 nguon va hop nhat thanh mot bang phuc vu bai toan du bao muc luong.
- Buoc cleaning giup chuan hoa salary, dia diem, kinh nghiem, suy ra category va loai bo duplicate.
- `jobs_cleaned.csv` giu du lieu da lam sach de phuc vu EDA.
- `jobs_features.csv` bo sung cac feature tu skill de san sang cho modeling.
- Cac phan truc quan hoa va dien giai insight chi tiet se duoc bo sung tiep theo.